# INITIALIZATION OF THE LOAN PREDICTION MODEL
This data set would provide you enough taste of working on data sets from insurance companies, what challenges are faced, what strategies are used, which variables influence the outcome etc. This is a classification problem. 

This is used to automate the loan eligibility process (real time) based on customer detail provided while filling online application form. 

In [19]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import os


breast_cancer_df = pd.read_csv(r'..\Training Datasets\Breast_cancer.csv')
breast_cancer_df.info()
print(os.getcwd())

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 33 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   id                       569 non-null    int64  
 1   diagnosis                569 non-null    str    
 2   radius_mean              569 non-null    float64
 3   texture_mean             569 non-null    float64
 4   perimeter_mean           569 non-null    float64
 5   area_mean                569 non-null    float64
 6   smoothness_mean          569 non-null    float64
 7   compactness_mean         569 non-null    float64
 8   concavity_mean           569 non-null    float64
 9   concave points_mean      569 non-null    float64
 10  symmetry_mean            569 non-null    float64
 11  fractal_dimension_mean   569 non-null    float64
 12  radius_se                569 non-null    float64
 13  texture_se               569 non-null    float64
 14  perimeter_se             569 non-null

# DATA CLEANING

In [20]:
# 1. Drop the ID column and the accidental empty 33rd column
breast_cancer_df = breast_cancer_df.drop(columns=['id', 'Unnamed: 32'], errors='ignore')

In [21]:
# CHECKING IF THERE ARE ANY DUPLICATES AND REMOVING THEM

print("No. of duplicated values are: ",breast_cancer_df.duplicated().sum(),'\n')
# CHECKING THE NO. OF NULL VALUES IN THE FEATURES
print("Columns with the number of null values:\n",breast_cancer_df.isnull().sum())


No. of duplicated values are:  0 

Columns with the number of null values:
 diagnosis                  0
radius_mean                0
texture_mean               0
perimeter_mean             0
area_mean                  0
smoothness_mean            0
compactness_mean           0
concavity_mean             0
concave points_mean        0
symmetry_mean              0
fractal_dimension_mean     0
radius_se                  0
texture_se                 0
perimeter_se               0
area_se                    0
smoothness_se              0
compactness_se             0
concavity_se               0
concave points_se          0
symmetry_se                0
fractal_dimension_se       0
radius_worst               0
texture_worst              0
perimeter_worst            0
area_worst                 0
smoothness_worst           0
compactness_worst          0
concavity_worst            0
concave points_worst       0
symmetry_worst             0
fractal_dimension_worst    0
dtype: int64


In [22]:
breast_cancer_df.columns.tolist()

['diagnosis',
 'radius_mean',
 'texture_mean',
 'perimeter_mean',
 'area_mean',
 'smoothness_mean',
 'compactness_mean',
 'concavity_mean',
 'concave points_mean',
 'symmetry_mean',
 'fractal_dimension_mean',
 'radius_se',
 'texture_se',
 'perimeter_se',
 'area_se',
 'smoothness_se',
 'compactness_se',
 'concavity_se',
 'concave points_se',
 'symmetry_se',
 'fractal_dimension_se',
 'radius_worst',
 'texture_worst',
 'perimeter_worst',
 'area_worst',
 'smoothness_worst',
 'compactness_worst',
 'concavity_worst',
 'concave points_worst',
 'symmetry_worst',
 'fractal_dimension_worst']

In [23]:
error_columns = [
    'radius_se', 'texture_se', 'perimeter_se', 'area_se', 'smoothness_se',
    'compactness_se', 'concavity_se', 'concave points_se', 'symmetry_se', 'fractal_dimension_se'
]

worst_case_columns = [
    'radius_worst', 'texture_worst', 'perimeter_worst', 'area_worst', 'smoothness_worst',
    'compactness_worst', 'concavity_worst', 'concave points_worst', 'symmetry_worst', 'fractal_dimension_worst'
]

# 2. Combine all lists into one single master list of columns to drop
all_redundant_columns = error_columns + worst_case_columns 

# 3. Drop them all in one clean step
# errors='ignore' ensures the code doesn't crash if a column name is slightly misspelled
breast_cancer_df = breast_cancer_df.drop(columns=all_redundant_columns, errors='ignore')


In [24]:
breast_cancer_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 569 entries, 0 to 568
Data columns (total 11 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   diagnosis               569 non-null    str    
 1   radius_mean             569 non-null    float64
 2   texture_mean            569 non-null    float64
 3   perimeter_mean          569 non-null    float64
 4   area_mean               569 non-null    float64
 5   smoothness_mean         569 non-null    float64
 6   compactness_mean        569 non-null    float64
 7   concavity_mean          569 non-null    float64
 8   concave points_mean     569 non-null    float64
 9   symmetry_mean           569 non-null    float64
 10  fractal_dimension_mean  569 non-null    float64
dtypes: float64(10), str(1)
memory usage: 49.6 KB


In [25]:
# CHECKING THE UNIQUE CATEGORICAL VALUES IN THE TARGET
breast_cancer_df['diagnosis'].unique()


<ArrowStringArray>
['M', 'B']
Length: 2, dtype: str

M - Malignant (Cancerous)
B - Benign (Non-cancerous)

In [26]:
# Encode target Feature
# M - 1(True)
# B - 0(False)
breast_cancer_df['diagnosis'] = breast_cancer_df['diagnosis'].map({'M': 1, 'B': 0})
breast_cancer_df.head(9)

,diagnosis,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,concave points_mean,symmetry_mean,fractal_dimension_mean
0,1,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,0.07871
1,1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667
2,1,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,0.05999
3,1,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,0.09744
4,1,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,0.05883
5,1,12.45,15.70,82.57,477.1,0.12780,0.17000,0.15780,0.08089,0.2087,0.07613
6,1,18.25,19.98,119.60,1040.0,0.09463,0.10900,0.11270,0.07400,0.1794,0.05742
7,1,13.71,20.83,90.20,577.9,0.11890,0.16450,0.09366,0.05985,0.2196,0.07451
8,1,13.00,21.82,87.50,519.8,0.12730,0.19320,0.18590,0.09353,0.2350,0.07389


SPLITTING OF THE DATASET COMES BEFORE SCALING OF THE FEATURES

The Correct Workflow:
Split your data into training and testing sets .

Fit and Transform the scaler on the training set only (scaler.fit_transform(X_train)).

Transform the test set using that same scaler (scaler.transform(X_test))—do not re-fit.

1. SPLITTING THE DATASET

In [29]:
# SPLITTING THE  breast_cancer_df DATASET INTO THE TRAINING AND TESTING SECTIONS

# SPLITTING IT FIRST INTO TRAINING SET AND TEST SET FIRST (80%- training and validation, 20%- testing)
from sklearn.model_selection import train_test_split


train_df,  test_df = train_test_split(breast_cancer_df, test_size=0.2, random_state=42)

print('Training dataframe shape: ', train_df.shape, '\n')
print('Testing dataframe shape: ', test_df.shape, '\n')

Training dataframe shape:  (455, 11) 

Testing dataframe shape:  (114, 11) 

